# PSD Normalization Example

This notebook aims to showcase how to normaliza PSD using the functionalities in the repo.

In [2]:
import os

import polars as pl
import numpy as np
import datetime as dt
from datetime import timedelta

from orcasound_noise.utils import Hydrophone, S3FileConnector
from orcasound_noise.analysis.partitioned_accessor import PartitionedAccessor

In [3]:
hydrophone = Hydrophone.ORCASOUND_LAB
os.environ["AWS_PROFILE"] = "ambient-sound-team" ## User will need tokens to access the S3 bucket. 

## Method 1: Normalization by a Scalar

In [4]:
# Step 1: Get the reference table 

## Set input time
date = dt.datetime(2026, 3, 9) # target date

## Access the refernce table
file_connector = S3FileConnector(hydrophone)
folder = 'data/temp/'
if not os.path.exists(folder):
    os.makedirs(folder)
ref_file_path = f"{folder}{hydrophone.name}_ref.parquet"

file_connector.get_ref_file(hydrophone, folder)
pl_ref = pl.read_parquet(ref_file_path)
pl_ref.head()

## Get the reference value of the date, we have saved the reference broadband, which calculated based on rolling 7-day 5th percentile of the broadband, in our s3
## Refer to orca-action-workflow repo for more details
bb_ref = pl_ref.filter(pl.col("date") == date).select('bb_ref').item()
bb_ref

-55.695

In [5]:
# Step 2: Get the PSD table
start = dt.datetime.combine(date, dt.time.min)
end = dt.datetime.combine(date, dt.time.max)

## Get the data
accessor = PartitionedAccessor(hydrophone, start, end)
pl_psd, pl_bb = accessor.get_dataframes(lazy=False)
pl_psd.head()

ind,67,71,75,80,85,90,95,100,106,112,118,125,132,140,150,160,170,180,190,200,212,224,236,250,265,280,300,315,335,355,375,400,425,450,475,500,…,2800,3000,3150,3350,3550,3750,4000,4250,4500,4750,5000,5300,5600,6000,6300,6700,7100,7500,8000,8500,9000,9500,10000,10600,11200,11800,12500,13200,14000,15000,16000,17000,18000,19000,20000,21200,22400
datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-03-09 00:00:00,-39.158099,-37.571866,-38.533838,-37.620634,-35.093424,-36.872626,-35.980675,-37.816728,-36.109695,-38.206514,-38.240449,-37.452902,-36.308716,-36.394764,-35.809287,-36.935891,-37.235352,-37.489158,-37.32393,-35.496143,-34.825202,-35.599633,-36.34443,-36.259068,-35.517053,-34.764925,-35.796271,-36.369816,-36.029954,-36.624168,-35.888645,-35.122638,-36.019625,-36.096674,-35.026674,-35.359068,…,-34.826674,-35.531061,-35.344352,-34.545202,-33.600046,-34.042701,-34.167474,-35.752235,-35.943478,-35.265894,-35.043829,-35.396851,-35.882095,-35.820893,-35.878747,-36.184299,-35.765266,-34.95968,-34.748093,-35.091061,-35.343372,-34.63692,-34.668943,-33.08692,-33.414299,-34.672808,-35.977474,-37.55774,-39.057744,-39.164168,-37.619586,-40.203201,-49.655655,-57.433961,-59.574859,-59.899068,-59.874974
2026-03-09 00:00:01,-38.645119,-38.339266,-36.730835,-36.490602,-36.743192,-36.580325,-35.431281,-35.905706,-36.031876,-36.674514,-35.606281,-35.875696,-34.435119,-35.667405,-34.960325,-34.786728,-33.607957,-33.224569,-33.730588,-34.338027,-33.550849,-33.824997,-34.798741,-34.812092,-34.591385,-34.483731,-34.759896,-35.264925,-33.163586,-33.960154,-35.938342,-34.691385,-34.619421,-35.291311,-34.718342,-34.966137,…,-35.262902,-35.318645,-35.536299,-35.236728,-34.74692,-34.486705,-34.333478,-35.364925,-35.66692,-35.619954,-35.956299,-36.007062,-36.638342,-35.978055,-36.247053,-36.434859,-36.404258,-35.721475,-34.974168,-35.367053,-35.781866,-35.35258,-34.730675,-33.670624,-34.145396,-35.116271,-36.554997,-38.354928,-39.307518,-39.508849,-37.617744,-40.373201,-48.914285,-57.17098,-59.439988,-59.844514,-59.849954
2026-03-09 00:00:02,-37.79475,-36.365416,-37.231719,-33.942497,-34.553275,-33.800893,-34.59427,-35.136271,-36.297258,-36.60222,-36.939712,-35.780379,-34.710046,-35.481605,-35.044928,-34.681905,-34.544925,-33.01472,-32.78795,-33.245944,-33.515951,-33.611447,-33.677258,-32.951705,-33.638092,-34.50815,-34.743043,-35.351166,-34.434258,-34.690123,-33.984989,-34.294989,-34.245202,-35.357957,-32.653961,-33.913201,…,-34.994437,-34.890046,-34.911201,-34.503471,-34.189196,-33.987606,-33.868027,-35.522095,-35.377345,-35.442564,-35.937455,-36.245416,-36.518645,-36.244928,-36.412902,-36.389068,-36.242902,-35.607053,-35.095912,-35.308342,-35.909816,-35.572235,-35.022235,-34.037053,-34.124925,-34.954767,-36.508607,-38.483913,-39.125396,-39.584652,-37.624997,-40.52258,-49.771866,-57.589263,-59.514767,-59.864767,-59.864974
2026-03-09 00:00:03,-37.252115,-36.91213,-37.594621,-35.842231,-34.190994,-34.536008,-34.409262,-34.622115,-34.5116,-35.175518,-36.434989,-35.515891,-36.412988,-35.724465,-34.80827,-34.768027,-34.549082,-33.829454,-33.881718,-33.875792,-34.228716,-35.27393,-35.640325,-34.858346,-35.189729,-35.877075,-35.424898,-35.584243,-35.930893,-35.236214,-34.770182,-34.216914,-34.38443,-34.258093,-34.62222,-34.943291,…,-34.167258,-35.230123,-34.664898,-34.221475,-33.434859,-33.829263,-33.946299,-35.257518,-35.548221,-35.254925,-35.134352,-35.249816,-35.675396,-35.628055,-35.719712,-36.12,-35.804352,-35.309586,-34.847053,-35.244168,-35.29741,-34.958093,-34.340154,-33.1816,-33.334997,-34.519068,-36.03258,-37.840675,-38.643372,-39.127518,-37.576674,-40.478849,-49.537062,-57.548849,-59.554974,-59.799896,-59.799712
2026-03-09 00:00:04,-38.068395,-38.216896,-39.808916,-39.205119,-38.266705,-37.344

In [6]:
# Step 3-1: Get the reference value of each frequency band
# Using white noise reference: the power is equally distributed across the entire frequency range (0-24 kHz)
import numpy as np

## Set parameters
total_freq_range = 24000 # Total frequency range
octave_bands = 12 # Number of octave bands

## Transform the reference broadband dB level to the Reference Power Density (Linear)
ref_power_density = (10**(bb_ref / 10)) 
ref_power_density /= total_freq_range

## Get the center frequencies of the octave bands
center_freqs = pl_psd.collect_schema().names()
center_freqs.remove('ind') # Remove timestamp column
center_freqs = np.array([int(freq) for freq in center_freqs])

## Width of a 1/12 octave band:
band_widths = center_freqs * (2**(1/(2*octave_bands)) - 2**(-1/(2*octave_bands)))

## Convert each band's reference to dB
spectral_ref_vector = 10 * np.log10(ref_power_density * band_widths)
spectral_ref_vector

array([-93.6193185 , -93.36748304, -93.12945389, -92.84916666,
       -92.58587727, -92.33764143, -92.10283047, -91.88006653,
       -91.62700788, -91.3878863 , -91.16124645, -90.9109664 ,
       -90.67432722, -90.41878617, -90.11915394, -89.8388667 ,
       -89.57557731, -89.32734148, -89.09253052, -88.86976657,
       -88.61670792, -88.37758634, -88.1509465 , -87.90066644,
       -87.64760779, -87.40848621, -87.10885398, -86.89696099,
       -86.62961846, -86.377783  , -86.13975385, -85.85946661,
       -85.59617723, -85.34794139, -85.11313043, -84.89036648,
       -84.63730783, -84.39818626, -84.09855402, -83.88666103,
       -83.6193185 , -83.36748304, -83.12945389, -82.84916666,
       -82.58587727, -82.33764143, -82.10283047, -81.88006653,
       -81.62700788, -81.3878863 , -81.16124645, -80.9109664 ,
       -80.67432722, -80.41878617, -80.11915394, -79.8388667 ,
       -79.57557731, -79.32734148, -79.09253052, -78.86976657,
       -78.61670792, -78.37758634, -78.1509465 , -77.90

In [7]:
# Step 4-1: Minus the reference value from the PSD table to get the normalized PSD using white noise reference.
df_normalized = pl_psd.with_columns([
    (pl.col(str(freq)) - spectral_ref_vector[i]).alias(str(freq)) for i, freq in enumerate(center_freqs)
])
df_normalized.head()

ind,67,71,75,80,85,90,95,100,106,112,118,125,132,140,150,160,170,180,190,200,212,224,236,250,265,280,300,315,335,355,375,400,425,450,475,500,…,2800,3000,3150,3350,3550,3750,4000,4250,4500,4750,5000,5300,5600,6000,6300,6700,7100,7500,8000,8500,9000,9500,10000,10600,11200,11800,12500,13200,14000,15000,16000,17000,18000,19000,20000,21200,22400
datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-03-09 00:00:00,54.461219,55.795617,54.595616,55.228533,57.492453,55.465015,56.122155,54.063339,55.517313,53.181372,52.920798,53.458064,54.365611,54.024022,54.309867,52.902976,52.340225,51.838184,51.768601,53.373623,53.791506,52.777953,51.806516,51.641599,52.130554,52.643561,51.312583,50.527145,50.599665,49.753615,50.251109,50.736828,49.576552,49.251268,50.086457,49.531299,…,42.581813,41.577793,41.552609,42.084416,42.777737,42.097052,41.691992,39.843943,39.404464,39.847237,39.846537,39.240457,38.516092,38.277661,38.007914,37.43502,37.602217,38.169774,38.101074,37.494816,36.99427,37.46591,37.211123,38.540088,37.973587,36.488438,34.933492,33.116587,31.361042,30.954986,32.219281,29.372376,19.671686,11.658569,9.294908,8.71764,8.502612
2026-03-09 00:00:01,54.974199,55.028217,56.398619,56.358565,55.842686,55.757317,56.671549,55.974361,55.595132,54.713373,55.554965,55.03527,56.239208,54.751381,55.158829,55.052139,55.96762,56.102772,55.361942,54.531739,55.065859,54.552589,53.352205,53.088574,53.056223,52.924755,52.348958,51.632036,53.466032,52.417629,50.201412,51.168082,50.976756,50.05663,50.394788,49.924229,…,42.145584,41.790209,41.360662,41.392891,41.630863,41.653049,41.525989,40.231252,39.681021,39.493176,38.934067,38.630246,37.759844,38.120499,37.639608,37.18446,36.963225,37.407979,37.874998,37.218824,36.555775,36.750251,37.149391,37.956384,37.24249,36.044976,34.355969,32.319399,31.111268,30.610305,32.221123,29.202376,20.413057,11.92155,9.429778,8.772194,8.527632
2026-03-09 00:00:02,55.824569,57.002067,55.897735,58.90667,58.032602,58.536748,57.508561,56.743796,55.32975,54.785666,54.221534,55.130587,55.964281,54.937181,55.074226,55.156962,55.030653,56.312621,56.304581,55.623822,55.100757,54.766139,54.473689,54.948962,54.009516,52.900336,52.365811,51.545795,52.195361,51.68766,52.154765,51.564478,51.350975,49.989984,52.459169,50.977165,…,42.414049,42.218808,41.98576,42.126147,42.188587,42.152147,41.991439,40.074083,39.970597,39.670567,38.952911,38.391892,37.879541,37.853626,37.473759,37.230251,37.124581,37.522401,37.753255,37.277535,36.427826,36.530596,36.857832,37.589955,37.262962,36.20648,34.402359,32.190415,31.29339,30.534502,32.21387,29.052997,19.555475,11.503267,9.355,8.751941,8.512612
2026-03-09 00:00:03,56.367203,56.455353,55.534833,57.006936,58.394883,57.801633,57.693568,57.257951,57.115408,56.212368,54.726258,55.395075,54.261339,54.694322,55.310884,55.070839,55.026495,55.497888,55.210812,54.993975,54.387992,53.103657,52.510622,53.042321,52.457879,51.531411,51.683956,51.312718,50.698725,51.141569,51.369572,51.642553,51.211747,51.089848,50.49091,49.947076,…,43.241228,41.878731,42.232063,42.408143,42.942924,42.310491,41.913167,40.338659,39.79972,39.858206,39.756014,39.387492,38.72279,38.470499,38.166949,37.499319,37.563131,37.819868,38.002113,37.341709,37.040231,37.144737,37.539913,38.445408,38.052889,36.642179,34.878387,32.833652,31.775414,30.991636,32.262193,29.096729,19.79028,11.543682,9.314792,8.816812,8.577874
2026-03-09 00:00:04,55.550923,55.150587,53.320538,53.644047,54.319173,54.993021,54.984223,54.888762,52.979779,53.223119,51.727972,53.276069,54.028643,52.24369,51.867341,53.128395,53.859785,53.582123,54.15999,53.975146,53.913695,52.26509,52.016578,52.277066,50.814595,51.347277,51.099125,51.076973,49.732767,51.016502,51.874552,52.324021,51.479663,49.838678,49.394187,48.747558,…

In [8]:
# Step 3-2: Get the reference value of each frequency band
# Using Pink noise reference: the power is equally distributed across the octave bands

## Set parameters
total_freq_range = 24000 # Total frequency range
octave_bands = 12 # Number of octave bands

## Get the center frequencies of the octave bands
center_freqs = pl_psd.collect_schema().names()
center_freqs.remove('ind') # Remove timestamp column
center_freqs = np.array([int(freq) for freq in center_freqs])

## Convert each band's reference to dB
## ref_per_band = 10 * log_{10}(10**(bb_ref / 10) / N) 
##              = 10 * log_{10}(10**(bb_ref / 10)) - 10 * log_{10}(N) 
##              = bb_ref - 10 * log_{10}(N)
ref_per_band = bb_ref - (10 * np.log10(len(center_freqs)))
ref_per_band

np.float64(-75.78100171761918)

In [9]:
# Step 4-2: Minus the reference value from the PSD table to get the normalized PSD using pink noise reference.
df_normalized = pl_psd.with_columns([
    (pl.col(str(freq)) - ref_per_band).alias(str(freq)) for i, freq in enumerate(center_freqs)
])
df_normalized.head()

ind,67,71,75,80,85,90,95,100,106,112,118,125,132,140,150,160,170,180,190,200,212,224,236,250,265,280,300,315,335,355,375,400,425,450,475,500,…,2800,3000,3150,3350,3550,3750,4000,4250,4500,4750,5000,5300,5600,6000,6300,6700,7100,7500,8000,8500,9000,9500,10000,10600,11200,11800,12500,13200,14000,15000,16000,17000,18000,19000,20000,21200,22400
datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-03-09 00:00:00,36.622903,38.209135,37.247164,38.160368,40.687578,38.908376,39.800326,37.964274,39.671307,37.574487,37.540553,38.3281,39.472285,39.386238,39.971714,38.845111,38.54565,38.291844,38.457072,40.284858,40.9558,40.181368,39.436572,39.521934,40.263948,41.016077,39.984731,39.411186,39.751048,39.156833,39.892357,40.658363,39.761376,39.684328,40.754328,40.421934,…,40.954328,40.249941,40.436649,41.2358,42.180956,41.7383,41.613527,40.028767,39.837524,40.515108,40.737172,40.38415,39.898907,39.960109,39.902254,39.596703,40.015736,40.821322,41.032909,40.689941,40.43763,41.144082,41.112059,42.694082,42.366703,41.108193,39.803527,38.223262,36.723258,36.616833,38.161416,35.5778,26.125346,18.347041,16.206143,15.881934,15.906028
2026-03-09 00:00:01,37.135883,37.441735,39.050167,39.2904,39.03781,39.200677,40.349721,39.875296,39.749126,39.106488,40.17472,39.905306,41.345883,40.113596,40.820677,40.994274,42.173044,42.556433,42.050413,41.442974,42.230153,41.956005,40.98226,40.968909,41.189617,41.297271,41.021105,40.516077,42.617415,41.820848,39.842659,41.089617,41.161581,40.489691,41.062659,40.814864,…,40.5181,40.462357,40.244703,40.544274,41.034082,41.294297,41.447524,40.416077,40.114082,40.161048,39.824703,39.77394,39.142659,39.802947,39.533948,39.346143,39.376744,40.059527,40.806833,40.413948,39.999135,40.428422,41.050326,42.110378,41.635605,40.664731,39.226005,37.426074,36.473484,36.272153,38.163258,35.4078,26.866717,18.610022,16.341013,15.936488,15.931048
2026-03-09 00:00:02,37.986252,39.415586,38.549283,41.838505,41.227727,41.980109,41.186732,40.644731,39.483744,39.178782,38.84129,40.000622,41.070956,40.299396,40.736074,41.099097,41.236077,42.766281,42.993052,42.535058,42.26505,42.169554,42.103744,42.829297,42.14291,41.272852,41.037959,40.429836,41.346744,41.090878,41.796013,41.486013,41.5358,40.423044,43.127041,41.8678,…,40.786564,40.890956,40.869801,41.27753,41.591805,41.793395,41.912974,40.258907,40.403657,40.338438,39.843546,39.535586,39.262357,39.536074,39.3681,39.391934,39.5381,40.173948,40.68509,40.472659,39.871186,40.208767,40.758767,41.743948,41.656077,40.826235,39.272395,37.297089,36.655605,36.19635,38.156005,35.258422,26.009135,18.191739,16.266235,15.916235,15.916028
2026-03-09 00:00:03,38.528886,38.868872,38.186381,39.938771,41.590008,41.244993,41.371739,41.158886,41.269402,40.605483,39.346013,40.265111,39.368014,40.056537,40.972732,41.012974,41.23192,41.951548,41.899284,41.90521,41.552285,40.507072,40.140677,40.922656,40.591273,39.903927,40.356104,40.196758,39.850109,40.544787,41.01082,41.564088,41.396572,41.522909,41.158782,40.837711,…,41.613744,40.550878,41.116104,41.559527,42.346143,41.951739,41.834703,40.523484,40.23278,40.526077,40.646649,40.531186,40.105605,40.152947,40.06129,39.661002,39.976649,40.471416,40.933948,40.536833,40.483592,40.822909,41.440848,42.599402,42.446005,41.261934,39.748422,37.940326,37.13763,36.653484,38.204328,35.302153,26.24394,18.232153,16.226028,15.981105,15.98129
2026-03-09 00:00:04,37.712606,37.564106,35.972086,36.575883,37.514297,38.436381,38.662395,38.789697,37.133773,37.616235,36.347727,38.146104,39.135318,37.605905,37.529188,39.07053,40.065209,40.035783,40.848461,40.886381,41.077989,39.668505,39.646633,40.157401,38.947989,39.719793,39.771273,39.961013,38.88415,40.419721,41.5158,42.245557,41.664487,40.271739,40.062059,39.638193,…,43.032029

## Method 2: Normalization by a Spectrum

In [10]:
# Step 1: Get the PSD table

## Set input time
date = dt.datetime(2026, 3, 9) # target date
start = dt.datetime.combine(date - dt.timedelta(7), dt.time.min) # 2026-03-02
end = dt.datetime.combine(date, dt.time.max) # 2026-03-09

## Get the data
accessor = PartitionedAccessor(hydrophone, start, end)
pl_psd, pl_bb = accessor.get_dataframes(lazy=True)

In [11]:
# Step 2: Calculate the reference value for each center frequency

import numpy as np

## get frequency column names
freqs = pl_psd.collect_schema().names() 
freqs.remove('ind')

## Calculate the reference sound level based on the reference period
## Reference period is 2026-03-02 to 2026-03-08 (7 days before the target date) in this example
ref_start = start
ref_end = dt.datetime.combine(date - dt.timedelta(1), dt.time.max) # 2026-03-08

pl_psd_ref = (
    pl_psd
    .filter(pl.col("ind").is_between(ref_start, ref_end))
    .select(
        pl.all().exclude("ind").quantile(0.05)
    )
).collect()

ref_dict = pl_psd_ref.to_dicts()[0]
ref_dict

{'67': -44.54170455173588,
 '71': -44.71120099038178,
 '75': -44.7689875113923,
 '80': -44.788921936696525,
 '85': -44.77896563591025,
 '90': -44.672308580740335,
 '95': -44.6208931116042,
 '100': -44.545220498140914,
 '106': -44.478054606089295,
 '112': -44.28520196660281,
 '118': -43.76120884135692,
 '125': -43.91347759350728,
 '132': -43.81696277602523,
 '140': -43.7,
 '150': -43.550324816587406,
 '160': -43.37906751977733,
 '170': -43.21968032258457,
 '180': -41.89730052689051,
 '190': -42.769435891076846,
 '200': -42.56485896818967,
 '212': -42.34489760731486,
 '224': -42.115792797442566,
 '236': -41.992221906943634,
 '250': -41.893528049061224,
 '265': -41.75774385732167,
 '280': -41.63943589107684,
 '300': -40.25200256555757,
 '315': -41.52159999873053,
 '335': -41.51568406482393,
 '355': -41.49587392869641,
 '375': -41.55256372492811,
 '400': -41.631605304378326,
 '425': -41.12835106972622,
 '450': -41.811311128853426,
 '475': -41.99016348873009,
 '500': -42.12971218322152,
 '5

In [12]:
# Step 3: Subtract the reference sound level from the PSD of each center band.
pl_psd_normalized = (
    pl_psd
    .filter(pl.col("ind").dt.date() == date.date())
    .with_columns([
        (pl.col(f) - ref_dict[f]).alias(f) 
        for f in freqs
    ])
    .collect()
)

pl_psd_normalized.head()


ind,67,71,75,80,85,90,95,100,106,112,118,125,132,140,150,160,170,180,190,200,212,224,236,250,265,280,300,315,335,355,375,400,425,450,475,500,…,2800,3000,3150,3350,3550,3750,4000,4250,4500,4750,5000,5300,5600,6000,6300,6700,7100,7500,8000,8500,9000,9500,10000,10600,11200,11800,12500,13200,14000,15000,16000,17000,18000,19000,20000,21200,22400
datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-03-09 00:00:00,5.383605,7.139335,6.23515,7.168288,9.685542,7.799682,8.640218,6.728493,8.36836,6.078688,5.52076,6.460575,7.508247,7.305236,7.741037,6.443177,5.984328,4.408143,5.445506,7.068716,7.519696,6.516159,5.647792,5.634461,6.24069,6.874511,4.455732,5.151784,5.48573,4.871706,5.663919,6.508967,5.108726,5.714638,6.96349,6.770645,…,4.224638,3.754833,3.843391,4.306664,5.556682,5.221736,4.767292,3.862624,3.65145,4.407008,4.929648,4.78688,3.927859,3.598819,3.231207,2.829179,3.080418,4.125087,4.356766,3.9729,3.590541,4.055592,3.410323,4.141422,3.776208,3.510393,3.782341,3.550867,3.273869,3.901744,5.655412,3.762644,3.099112,1.925993,0.47,0.140886,0.08
2026-03-09 00:00:01,5.896585,6.371935,8.038153,8.29832,8.035774,8.091984,9.189612,8.639515,8.446179,7.610688,8.154927,8.037782,9.381844,8.032595,8.59,8.59234,9.611723,8.672732,9.038847,8.226832,8.794049,8.290796,7.19348,7.081436,7.166359,7.155705,5.492106,6.256675,8.352098,7.53572,5.614221,6.94022,6.50893,6.52,7.271821,7.163575,…,3.788409,3.967249,3.651445,3.615138,4.409808,4.777733,4.601289,4.249934,3.928008,4.052948,4.017178,4.176669,3.171612,3.441658,2.862901,2.578619,2.441426,3.363292,4.130691,3.696908,3.152046,3.339932,3.348591,3.557719,3.045111,3.06693,3.204819,2.753679,3.024094,3.557063,5.657253,3.592644,3.840482,2.188974,0.60487,0.19544,0.10502
2026-03-09 00:00:02,6.746955,8.345785,7.537269,10.846425,10.225691,10.871415,10.026623,9.40895,8.180797,7.682982,6.821497,8.133098,9.106917,8.218395,8.505397,8.697162,8.674756,8.88258,9.981486,9.318915,8.828946,8.504345,8.314964,8.941823,8.119652,7.131286,5.50896,6.170434,7.081426,6.80575,7.567575,7.336617,6.883149,6.453354,9.336202,8.216511,…,4.056874,4.395848,4.276543,4.348395,4.967532,5.276831,5.066739,4.092764,4.217583,4.230338,4.036022,3.938315,3.291309,3.174784,2.697052,2.62441,2.602782,3.477714,4.008947,3.755619,3.024097,3.120278,3.057032,3.191289,3.065582,3.228434,3.251209,2.624695,3.206216,3.48126,5.65,3.443265,2.982901,1.770691,0.530092,0.175187,0.09
2026-03-09 00:00:03,7.289589,7.799071,7.174367,8.946691,10.587972,10.1363,10.211631,9.923105,9.966455,9.109684,7.32622,8.397587,7.403975,7.975535,8.742055,8.61104,8.670598,8.067847,8.887718,8.689067,8.116181,6.841863,6.351897,7.035182,6.568015,5.762361,4.827105,5.937357,5.584791,6.25966,6.782382,7.414691,6.743921,7.553218,7.367943,7.186422,…,4.884053,4.05577,4.522846,4.630391,5.721869,5.435174,4.988468,4.357341,4.046707,4.417977,4.839125,4.933915,4.134557,3.791658,3.390242,2.893478,3.041332,3.775181,4.257806,3.819793,3.636502,3.734419,3.739113,4.046742,3.85551,3.664134,3.727236,3.267932,3.688241,3.938394,5.698324,3.486996,3.217705,1.811105,0.489885,0.240058,0.155262
2026-03-09 00:00:04,6.473309,6.494305,4.960072,5.583803,6.512261,7.327688,7.502286,7.553916,5.830826,6.120435,4.327934,6.27858,7.171279,5.524904,5.298511,6.668596,7.503888,6.152082,7.836895,7.670238,7.641885,6.003296,5.857854,6.269928,4.924731,5.578227,4.242274,5.701612,4.618833,6.134593,7.287362,8.09616,7.011837,6.302048,6.27122,5.986904,…,6.302338,5.055218,5.401445,5.734117,7.251283,7.05954,5.541239,4.935273,5.09212,5.743827,6.414532,5.944051,4.37221,4.395981,4.04933,3.403581,3.514209,4.488062,4.966638,4.45023,3.810711,4.303445,3.584914,3.999735,3.889196,4.015147,3.99416,3.670632,3.837182,3.726844,5.777944,3.533623,2.96937,1.855993,0.554905,0.109965,0.08
